# **Diffusion of Influence**

This notebook demonstrates the following models and algorithms:
1. Linear Threshold (LT) model  
3. Path-based influence sanity check (for LT)  
2. Independent Cascade (IC) model  
4. Greedy influence maximization (for IC)


## Imports

- `networkx` for graph structures and basic algorithms  
- `matplotlib` for visualization  
- `random` for stochastic threshold and activation sampling  

A fixed random seed (`random.seed(...)`) is set for reproducible Monte Carlo results.

In [ ]:
# imports
import networkx as nx
import random
import matplotlib.pyplot as plt
from itertools import combinations
from collections import defaultdict

# show plots inline
%matplotlib inline

random.seed(42)


## Demo graphs

To get an intuition for how influence spreads, we use a simple directed graph with weighted edges. Edge weights represent *influence strengths* (for LT) or *probabilities* (for IC).
This example is deliberately simple so you can inspect how activation propagates step by step.

As an alternative, you can try the *Karate Club* network. In this one, edge weights equal to the number of shared contexts (1-8), so we convert it to a directed graph and normalize its weights to $[0,1]$.

In [ ]:
# Toy graph
G_toy = nx.DiGraph()
edges = [
    ('A', 'B', 0.4), ('A', 'C', 0.6),
    ('B', 'D', 0.5), ('C', 'D', 0.3),
    ('C', 'E', 0.7)
]
G_toy.add_weighted_edges_from(edges)

# Visualize toy graph
pos = nx.spring_layout(G_toy, seed=42)
plt.figure(figsize=(5,4))
nx.draw(G_toy, pos, with_labels=True, node_size=800, arrows=True)
edge_labels = {(u,v): f"{d['weight']:.2f}" for u,v,d in G_toy.edges(data=True)}
nx.draw_networkx_edge_labels(G_toy, pos, edge_labels=edge_labels)
plt.title("Toy DiGraph")
plt.show()

In [ ]:
# Karate Club
G_karate_raw = nx.karate_club_graph()  # undirected with integer weights (1..8)

G_karate = nx.DiGraph()
for u, v, d in G_karate_raw.edges(data=True):
    p = d.get('weight', 1) / 8.0
    # add both directions
    G_karate.add_edge(u, v, weight=p)
    G_karate.add_edge(v, u, weight=p)


## **Linear Threshold (LT) model**

**Goal:** Simulate activation rounds on a directed, weighted graph until convergence.  

A node $v$ becomes active when the *total influence* it receives from its neighbors meets or exceeds its personal threshold:

$\sum_{w \in \mathcal{N}(v)} b_{v,w} \;\ge\; \theta_v$

Where:
- $w \in \mathcal{N}(v)$: neighbor nodes that can influence $v$
  (for directed graphs, these are the *in-neighbors* with edges $w \to v$).
- $b_{v,w}$: influence weight from $w$ to $v$, typically in $[0,1]$;
  often normalized so that $\sum_{w \in \mathcal{N}(v)} b_{v,w} \le 1$.
- $\theta_v$: activation threshold of $v$, drawn randomly from $[0,1]$ or set to a fixed value.

Once activated, nodes remain active. The process continues until no new activations occur.


In [ ]:
import networkx as nx
import random
import matplotlib.pyplot as plt

def simulate_LT(G, seeds, thresholds=None, max_steps=100, verbose=True):
    """
    Linear Threshold (LT) diffusion simulation.
    G: weighted directed graph (DiGraph)
    seeds: initial active nodes (list or set)
    thresholds: dict {node: theta in [0,1]} or random if None
    Returns:
        history: list of sets of active nodes per round
        influence: total number (or fraction) of activated nodes
    """
    active = set(seeds)
    history = [set(active)]
    if thresholds is None:
        thresholds = {v: random.random() for v in G.nodes()}

    for step in range(1, max_steps + 1):
        new_active = set(active)
        for v in G.nodes():
            if v in active:
                continue
            # Influence sum from active in-neighbors
            infl = sum(G[u][v].get('weight', 1.0) for u in G.predecessors(v) if u in active)
            if infl >= thresholds[v]:
                new_active.add(v)
        if new_active == active:
            break
        active = new_active
        history.append(set(active))

    influence = len(active) / len(G)  # Fraction of total nodes activated

    if verbose:
        print(f"Simulation finished in {len(history)-1} iterations.")
        print(f"Final activated nodes: {sorted(active)}")
        print(f"Total influence: {len(active)} / {len(G)} = {influence:.2f}")

    return history, influence


#### Try the LT model with some seed nodes

In [ ]:
# Set the graph to be used
G = G_toy
# G = G_karate

# Set the seeds
seeds = ['A','B']  # toy graph
# seeds = [0]  # karate club

# Simulate
history, influence = simulate_LT(G, seeds)

### **LT:** Complete search with Monte Carlo simulation

**best_k_LT**: Performs complete search over all *k*-node seed sets. For each candidate, it runs Monte Carlo simulations R times with new random thresholds $\sim U(0, 1)$ and averages the final spread.

For demonstration, use small k and R values; increase R for more stable estimates.

---
**Helpers**

- **normaliz_weights**: Rescales all edge weights in G to $[0, 1]$ via global min-max normalization.

- **final_active_LT**: Runs one deterministic LT simulation and returns the final active set. Pass a fixed `thresholds` dictionary to remove randomness.


In [ ]:
import itertools
import random
import networkx as nx

def final_active_LT(G, seeds, thresholds, max_steps=100):
    history, _ = simulate_LT(G, seeds, thresholds=thresholds, max_steps=max_steps, verbose=False)
    return history[-1]

def best_k_LT(G, k, thresholds=None, R=500, max_steps=100):
    """
    Exhaustive search over all k-node seed sets under LT with random thresholds.
    For each seed set, run MC sampling R times with random thresholds ~ U(0,1).

    R: Number of Monte Carlo samples per seed set	200–1000
    k: Number of seeds
    max_steps: Max LT iterations (must be >= number of nodes)

    Returns:
        best_S: seed set giving the highest expected spread
        best_val: estimated expected spread (average #active)
        results: dict mapping each seed set -> average spread
    """
    best_S, best_val = None, -1.0
    results = {}
    nodes = list(G.nodes())

    for S in itertools.combinations(nodes, k):
        total_spread = 0
        for _ in range(R):
            A = final_active_LT(G, S, thresholds, max_steps=max_steps)
            total_spread += len(A)
        avg_spread = total_spread / R
        results[S] = avg_spread
        if avg_spread > best_val:
            best_val, best_S = avg_spread, S

    return best_S, best_val, results


### **LT:** Finding the best seed(s)

This example uses a complete search under random thresholds.

You can switch between the toy graph and the Karate Club network by commenting/uncommenting the lines below.

Results show the best seed set and a ranked list of all candidates.


In [ ]:
# Set the graph to be used
G = G_toy
# G = G_karate

best_S, best_val, table = best_k_LT(G, k=1, R=1000)
print(f"Best seed set: {best_S}  |  Expected spread ≈ {best_val:.2f}")

# Show all results sorted
for S, val in sorted(table.items(), key=lambda x: -x[1]):
    print(f"{S}: {val:.2f}")

### **LT:** Path-based influence ("sanity check" for LT)

Idea: On a small graph, approximate the influence from source → target as the *sum of path probabilities* (product of edge weights along each simple path), limited to short paths.  

Compare to the *empirical* probability that the target activates in the LT model with random thresholds.

In [ ]:
def all_simple_paths_limited(G, source, target, max_len=4):
    """Enumerate simple paths up to length max_len."""
    paths = []
    def dfs(path):
        if len(path) > max_len:
            return
        last = path[-1]
        if last == target:
            paths.append(path[:])
            return
        for nbr in G.successors(last):
            if nbr not in path:
                dfs(path + [nbr])
    dfs([source])
    return paths


def path_based_influence(G, source, target, max_len=4):
    """Approximate path influence as sum of path probabilities."""
    paths = all_simple_paths_limited(G, source, target, max_len)
    total_prob = 0.0
    for path in paths:
        prob = 1.0
        for u, v in zip(path[:-1], path[1:]):
            prob *= G[u][v].get('weight', 1.0)
        total_prob += prob
    return total_prob


In [ ]:
# Set the graph to be used
G = G_toy
source, target = 'A', 'E'
# G = G_karate
# source, target = 0, 33

# Compare path-based vs LT Monte Carlo
approx_infl = path_based_influence(G, source, target, max_len=5)
print(f"Path-based influence {source}→{target}: {approx_infl:.3f}")

# Rough check via LT Monte Carlo: fraction of runs where target activates

count = 0
runs = 500
for _ in range(runs):
    thresholds = {v: random.random() for v in G.nodes()}
    history, _ = simulate_LT(G, seeds=[source], thresholds=thresholds, verbose=False)
    if target in history[-1]:
        count += 1

print(f"Simulated P({target} activated | seed={source}): {count/runs:.3f}")

---
---


## **Independent Cascade (IC) model**

**Goal:** Simulate activation rounds on a directed, weighted graph until convergence.  

For each directed edge $(u, v)$, there is an activation probability $p_{u,v} \in [0,1]$.  
When node $u$ becomes active at time $t$, it attempts to activate every inactive neighbor $v \in \mathcal{N}^+(u)$ at the next step $t+1$:  

$P(v\ \text{becomes active at}\ t+1 \mid u\ \text{active at}\ t) = p_{u,v}$

Each activation attempt is *independent*, and each node gets only *one opportunity* to influence its neighbors. If an attempt fails, that edge cannot trigger activation later.

Notations:
- $v$: Target node potentially influenced by its active neighbors.  
- $\mathcal{N}^+(u)$: The out-neighbors of $u$, i.e. nodes with edges $u \to v$.  
- $p_{u,v}$: Activation probability along edge $(u,v)$, typically derived from observed influence strength or normalized weights.  

Note that activations are stochastic, i.e., repeated simulations can yield different cascades even with the same seed set. Once activated, nodes remain active (no deactivation). The process continues in discrete steps until no new nodes activate; guaranteed to terminate in at most $|V|$ iterations.

In [ ]:
def simulate_IC_once(G, seeds, use_edge_weights=True, p_default=0.1, max_steps=1000):
    """Run one IC cascade and return the activation history as a list of sets.
    - If use_edge_weights=True, use G[u][v]['weight'] as activation probability.
    - Otherwise, use p_default for every edge.
    """
    active = set(seeds)
    history = [set(active)]
    frontier = list(seeds)
    steps = 0

    while frontier and steps < max_steps:
        next_frontier = []
        for u in frontier:
            for v in G.successors(u):
                if v in active:
                    continue
                p = G[u][v].get('weight', p_default) if use_edge_weights else p_default
                if random.random() < p:
                    active.add(v)
                    next_frontier.append(v)
        if not next_frontier:
            break
        history.append(set(active))
        frontier = next_frontier
        steps += 1

    return history


### **IC:** Monte Carlo expected spread

We estimate expected influence (spread) by averaging the final active count over many independent IC runs.


In [ ]:
def expected_spread_IC_MC(G, seeds, R=500, use_edge_weights=True, p_default=0.1):
    """Monte Carlo estimate of E[final active count] under IC."""
    total = 0
    for _ in range(R):
        hist = simulate_IC_once(G, seeds, use_edge_weights=use_edge_weights, p_default=p_default)
        total += len(hist[-1])
    return total / R

# Set the graph to be used
G = G_toy
seeds = ['A']  # toy graph
# G = G_karate
# seeds = [0]  # karate club

spread_toy = expected_spread_IC_MC(G, seeds=seeds, R=300, use_edge_weights=True)
print(f"Expected spread: {spread_toy:.2f}")

### **IC:** Distribution of cascade sizes

Because IC is stochastic, final cascade sizes vary across runs.  
This histogram helps you see the *variance* and *skew* in outcomes for a fixed seed set.

In [ ]:
def spread_distribution(G, seeds, R=500, use_edge_weights=True, p_default=0.1):
    vals = []
    for _ in range(R):
        hist = simulate_IC_once(G, seeds, use_edge_weights=use_edge_weights, p_default=p_default)
        vals.append(len(hist[-1]))
    return vals

# Set the graph to be used (toy or karate club)
vals = spread_distribution(G_toy, seeds=['A'], R=400, use_edge_weights=True)  # toy graph
# vals = spread_distribution(G_karate, seeds=[0], R=400, use_edge_weights=True)  # karate club

plt.figure(figsize=(6,4))
plt.hist(vals, bins=min(15, len(set(vals))))
plt.title("IC: Distribution of final cascade sizes")
plt.xlabel("Final active count")
plt.ylabel("Frequency")
plt.grid(alpha=0.3)
plt.show()

### **IC:** Greedy influence maximization

Picks the next node with the *largest marginal gain* in expected spread, estimated via Monte Carlo. This is near-optimal for submodular objectives.

In [ ]:
def greedy_seed_set_IC(G, k, R=300, use_edge_weights=True, p_default=0.1):
    """Greedy influence maximization under IC using Monte Carlo expected spread.
    Returns the selected seed list and a log of marginal gains."""
    S = []
    gains = []
    nodes = list(G.nodes())
    for i in range(k):
        best_v, best_val = None, -1.0
        base = expected_spread_IC_MC(G, S, R=R, use_edge_weights=use_edge_weights, p_default=p_default) if S else 0.0
        for v in nodes:
            if v in S:
                continue
            val = expected_spread_IC_MC(G, S + [v], R=R, use_edge_weights=use_edge_weights, p_default=p_default)
            marg = val - base
            if marg > best_val:
                best_val, best_v = marg, v
        S.append(best_v)
        gains.append(best_val)
        print(f"Step {i+1}: add {best_v} | marginal ≈ {best_val:.2f} | E[spread](S) ≈ {expected_spread_IC_MC(G, S, R=R, use_edge_weights=use_edge_weights, p_default=p_default):.2f}")
    return S, gains

# Pick top-2 seeds

# Set the graph to be used
G = G_toy
# G = G_karate

best_seeds_greedy, gains = greedy_seed_set_IC(G, k=2, R=2000, use_edge_weights=True)
print("Greedy seeds:", best_seeds_greedy)

### **IC:** Complete search over seed sets (small graphs)

For small networks and small $k$, you can exhaustively evaluate all $\binom{|V|}{k}$ seed sets by Monte Carlo simulation.  
This provides a *ground truth comparator* for the greedy method on small graphs.

In [ ]:
def best_k_IC_complete_MC(G, k, R=400, use_edge_weights=True, p_default=0.1):
    """Exhaustive search over all k-node seed sets using Monte Carlo expected spread.
    Returns: (best_set, best_value, results_dict)"""
    nodes = list(G.nodes())
    best_S, best_val = None, -1.0
    results = {}
    for S in combinations(nodes, k):
        val = expected_spread_IC_MC(G, list(S), R=R, use_edge_weights=use_edge_weights, p_default=p_default)
        results[S] = val
        if val > best_val:
            best_val, best_S = val, S
    return best_S, best_val, results

# Demo (toy graph): exhaustive best k=2
G = G_toy
best_S_exh, best_val_exh, table_exh = best_k_IC_complete_MC(G, k=2, R=2000, use_edge_weights=True)
print(f"Exhaustive best: {best_S_exh}  |  E[spread] ≈ {best_val_exh:.2f}")

# Compare to greedy result
print("Greedy vs Exhaustive:", best_seeds_greedy, "vs", best_S_exh)